# **01_bronze_layer_ingestion**

In [0]:
%python

#test 1: Does the file exist, and what is its exact name?
display(dbutils.fs.ls('/Volumes/topcars_sa/bronze/raw_files/'))

#What it does: Lists all files in the raw_files volume folder.
#Expected output: A table with columns path, name, size. You should see your CSV file (e.g. SA Car Sales - Sheet1.csv or _SA Car Sales - Sheet1.csv).
#✅ Confirms the file exists and gives the exact name/size.

In [0]:
%python

#test 2:What does the raw content actually look like?
with open('/Volumes/topcars_sa/bronze/raw_files/SA Car Sales - Sheet1.csv', 'r') as f:
    content = f.read()
print(repr(content[:300]))

#What it does: Reads the file as raw text and prints the first 300 characters with hidden characters visible.
#Expected output: 
# A string like: 'Sales_ID,Dealer,Model,Price\n1,ABC Motors,Toyota,250000\n2,...'

In [0]:
%python

#test 3: Can Spark read it into a proper DataFrame?
df = spark.read.option("header", True).option("inferSchema", True).csv('/Volumes/topcars_sa/bronze/raw_files/SA Car Sales - Sheet1.csv')
display(df)
print("Row count:", df.count())

#What it does: Loads the CSV into a Spark DataFrame with headers and inferred types.
#Expected output:

#display(df) → a table with column names as headers and sample rows.

#print(...) → something like Row count: 1500.
#✅ Confirms Spark can parse the file, the schema looks right, and gives you the total row count.



In [0]:
SELECT * FROM topcars_sa.bronze.dealer_bronze LIMIT 5;

In [0]:
SELECT * FROM topcars_sa.bronze.models_bronze LIMIT 5;

In [0]:
SELECT * FROM topcars_sa.bronze.sales_bronze LIMIT 5;

In [0]:
SELECT COUNT(*) FROM topcars_sa.bronze.dealer_bronze;

In [0]:
SELECT COUNT(*) FROM topcars_sa.bronze.models_bronze;

In [0]:
SELECT COUNT(*) FROM topcars_sa.bronze.sales_bronze;

# **02_silver_layer_transform**

In [0]:
SELECT * FROM topcars_sa.silver.models_silver LIMIT 5;

In [0]:
SELECT DealerID, COUNT(*) FROM topcars_sa.silver.dealer_silver GROUP BY DealerID HAVING COUNT(*) > 1;

In [0]:
SELECT * FROM topcars_sa.silver.dealer_silver WHERE dealerID IS NULL OR DealerName IS NULL;

In [0]:
-- Counting number of rows
SELECT COUNT(*) FROM topcars_sa.silver.sales_silver;

In [0]:
-- Show/display the top 5 rows.  
SELECT * FROM topcars_sa.silver.sales_silver LIMIT 5;

In [0]:
-- Check for duplicate SaleIDs
SELECT SaleID, COUNT(*) FROM topcars_sa.silver.sales_silver GROUP BY SaleID HAVING COUNT(*) > 1;

In [0]:
-- Check for any nulls in key colums
SELECT * FROM topcars_sa.silver.sales_silver 
WHERE SaleID IS NULL OR DealerID IS NULL OR ModelID IS NULL OR Date IS NULL;

In [0]:
-- ModelID missing.
DESCRIBE TABLE topcars_sa.silver.sales_silver;

In [0]:
SELECT DISTINCT DealerID 
FROM topcars_sa.silver.sales_silver
WHERE DealerID NOT IN (SELECT DealerID FROM topcars_sa.silver.dealer_silver);

In [0]:
-- Any ModelID in sales_silver that doesn't exist in models_silver?
SELECT DISTINCT ModelID 
FROM topcars_sa.silver.sales_silver
WHERE ModelID NOT IN (SELECT ModelID FROM topcars_sa.silver.models_silver);

# **03_gold_layer_star_schema**

In [0]:
SELECT * FROM topcars_sa.gold.dim_dealer LIMIT 10;


In [0]:
SELECT * FROM topcars_sa.gold.dim_model LIMIT 10;

In [0]:
SELECT * FROM topcars_sa.gold.dim_date LIMIT 10;

In [0]:
-- Row count — should be exactly 1,461
-- 365 + 365 + 366 (2024 leap) + 365 = 1461
SELECT COUNT(*) FROM topcars_sa.gold.dim_date;

In [0]:
-- No gaps or duplicates
-- All 3 should return 1461
SELECT 
    COUNT(*) AS total_rows,
    COUNT(DISTINCT FullDate) AS distinct_dates,
    COUNT(DISTINCT DateKey) AS distinct_keys
FROM topcars_sa.gold.dim_date;

In [0]:
-- Check for any nulls in key colums
-- All should be zero
SELECT
  SUM(CAST(FullDate  IS NULL AS INT)) AS null_date,
  SUM(CAST(DateKey   IS NULL AS INT)) AS null_key,
  SUM(CAST(MonthName IS NULL AS INT)) AS null_month_name,
  SUM(CAST(DayName   IS NULL AS INT)) AS null_day_name,
  SUM(CAST(Quarter   IS NULL AS INT)) AS null_quarter
FROM topcars_sa.gold.dim_date;

In [0]:
-- A quick sanity check
-- The count should match the sales_silver row count of 218
SELECT COUNT(*) FROM topcars_sa.gold.fact_sales;